In [1]:
import pandas as pd
import fitz
import cv2
from io import BytesIO
from PIL import Image
import base64
from openai import OpenAI
from pydantic import BaseModel
import json
import numpy as np
import traceback
from typing import Optional
import io

import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Alignment, Border, Side, Font

from blank_functions.paths import path_to_data
from blank_functions.forms.form_recognition import FormRecognition
from blank_functions.ui.ui_functions import get_pic_from_pdf, save_to_excel_local, get_correct_answers, postprocess_raw_output, check_answers, final_styling, extract_text_from_image, transform_json_to_dataframe
from blank_functions.ui.ui_functions import promt, prepare_cur_dict, reorder_cols

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# no limit columns for pandas
pd.set_option('display.max_columns', None)

/Users/vladislav/Documents/mom_project_repo/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/vladislav/Documents/mom_project_repo/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import time

import os
# /Users/vladislav/Documents/mom_project_repo/streamlit-exam-form-app/notebooks/docling_test.ipynb
os.chdir("/Users/vladislav/Documents/mom_project_repo/streamlit-exam-form-app/notebooks")
# pdf_path = "./data/valid_format/valid_questions.pdf"
# answers_path = "./data/answers.xlsx"
# template_path = "./data/template_2.jpg"
# json_path = "./data/rows_data.json"

pdf_path = f"{path_to_data}/valid_format/users_answears.pdf"
answers_path = f"{path_to_data}/answers.xlsx"
template_path = f"{path_to_data}/template_raw.jpg"
json_path = f"{path_to_data}/rows_data_new_format.json"



pdf_bytes = open(pdf_path, 'rb').read()
pdf_document = fitz.open(stream=pdf_bytes, filetype="pdf")
num_pages = pdf_document.page_count

answers_bytes = open(answers_path, 'rb').read()
answers = pd.read_excel(BytesIO(answers_bytes))

cur_version = 2

df_global = pd.DataFrame()
form_dict = {}
for i in range(0, num_pages):
    cur_pic = get_pic_from_pdf(pdf_bytes, i, zoom=6.0)
    form = FormRecognition(
        image = cur_pic,
        template_path = template_path,
        json_path = json_path,
        answers = answers,
        version = cur_version)
    form = form.run_pipeline()
    cur_dict = prepare_cur_dict(form)
    df_current = transform_json_to_dataframe(cur_dict)
    df_global = pd.concat([df_global, df_current]).reset_index(drop=True)
    form_dict[i] = form

correct_answers = get_correct_answers(answers_bytes)
df_global_processed = postprocess_raw_output(df_global, correct_answers, cur_version)
df_global_answers = check_answers(df_global_processed)
df_global_styled = final_styling(df_global_answers)
df_global_styled = reorder_cols(df_global_styled)
save_to_excel_local(df_global_styled, form_dict)


/Users/vladislav/Documents/mom_project_repo/streamlit-exam-form-app/blank_functions/ui/ui_functions.py:177: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  correct_answers = pd.read_excel(correct_answers_path)


In [3]:
correct_answers = get_correct_answers(answers_bytes)
df_global_processed = postprocess_raw_output(df_global, correct_answers, cur_version)
df_global_answers = check_answers(df_global_processed)
df_global_styled = final_styling(df_global_answers)
df_global_styled = reorder_cols(df_global_styled)
# save_to_excel_local(df_global_styled, form_dict)

/Users/vladislav/Documents/mom_project_repo/streamlit-exam-form-app/blank_functions/ui/ui_functions.py:177: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  correct_answers = pd.read_excel(correct_answers_path)


In [5]:
df_global_styled

,Дата,Картинка Дата,Код участника,Картинка код участника,Вариант,Картинка вариант,Задание 1,Картинка ответа 1,Задание 2,Картинка ответа 2,Задание 3,Картинка ответа 3,Задание 4,Картинка ответа 4,Задание 5,Картинка ответа 5,Задание 6,Картинка ответа 6,Задание 7,Картинка ответа 7,Задание 8,Картинка ответа 8,Задание 9,Картинка ответа 9,Задание 10,Картинка ответа 10,Начисленные баллы 1,Начисленные баллы 2,Начисленные баллы 3,Начисленные баллы 4,Начисленные баллы 5,Начисленные баллы 6,Начисленные баллы 7,Начисленные баллы 8,Начисленные баллы 9,Начисленные баллы 10,Начисленные баллы сумма
0,120325,,1293,,2,,280.0,,38.0,,14.0,,3478.0,,24.5,,102.0,,3.0,,-0.5,,17.05,,0.75,,0,0,1,1,1,1,0,1,0,0,5


In [ ]:
def postprocess_raw_output(df_global_fin, correct_answers, version):

    df_global_fin['Вариант'] = version

    df_global_fin['Дата'] = df_global_fin['Дата'].str.upper()
    df_global_fin['Вариант'] = df_global_fin['Вариант'].astype(int)
    df_global_fin['Вариант'] = df_global_fin['Вариант'].astype(str)
    for i in range(1, 11):
        df_global_fin[f'Задание {i}'] = df_global_fin[f'Задание {i}'].replace(',', '.', regex=True)
        df_global_fin[f'Задание {i}'] = df_global_fin[f'Задание {i}'].replace('', 'nan').astype(float)
        df_global_fin[f'Замена {i}'] = df_global_fin[f'Замена {i}'].replace(',', '.', regex=True)
        df_global_fin[f'Замена {i}'] = df_global_fin[f'Замена {i}'].replace('', 'nan').astype(float)
    for col in df_global_fin.columns:
        df_global_fin[col] = df_global_fin[col].astype(str)

    total_df = pd.merge(df_global_fin, correct_answers, on="Вариант", how="left")

    return total_df

correct_answers = get_correct_answers(answers_bytes)
df_global_processed = postprocess_raw_output(df_global, correct_answers, cur_version)
df_global_answers = check_answers(df_global_processed)
df_global_styled = final_styling(df_global_answers)
df_global_styled = reorder_cols(df_global_styled)
save_to_excel_local(df_global_styled, form_dict)

In [ ]:
from skimage import io, filters
from skimage.color import rgb2gray

img = io.imread("image.jpg", as_gray=True)
threshold = filters.threshold_otsu(img)  # Автоматический порог
binary = img > threshold  # Делаем маску
binary = binary.astype(np.int8)
io.imshow(binary)
io.show()

In [ ]:
from transformers import CLIPProcessor, CLIPModel, CLIPVisionModel
from PIL import Image
import torch
import matplotlib.pyplot as plt
import cv2
import numpy as np

rows = [(16, form_dict[16].user_id),
(4, form_dict[4].answer1),
(2, form_dict[2].answer5),
(11, form_dict[11].answer5),
(12, form_dict[12].answer5),
(12, form_dict[12].answer8),
(11, form_dict[11].answer9),
(16, form_dict[16].answer10)]


string_to_digit = {
    'comma': ',',
    'minus': '-',
    'one': '1',
    'two': '2',
    'three': '3',
    'four': '4',
    'five': '5',
    'six': '6',
    'seven': '7',
    'eight': '8',
    'nine': '9',
    'zero': '0'
}

from torchvision import transforms
from PIL import Image


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vision_model = CLIPVisionModel.from_pretrained('tanganke/clip-vit-base-patch32_mnist').to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_model.vision_model.load_state_dict(vision_model.vision_model.state_dict())

# Load reference images
ref_images_dict = {name: cv2.resize(cv2.imread(f"C:/Users/zamko/Documents/mom_project/repo/data/ref_pics/{name}.png"), (28, 28), interpolation=cv2.INTER_AREA)
                   for name in ['comma', 'minus', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'zero']}

for key, image in ref_images_dict.items():
    threshold = filters.threshold_otsu(image)  # Автоматический порог
    binary = image > threshold  # Делаем маску
    binary_cv2 = (binary * 255).astype(np.uint8)
    # binary_cv2 = cv2.cvtColor(binary_cv2, cv2.COLOR_GRAY2BGR)
    ref_images_dict[key] = binary_cv2


ref_embeddings = {}
for key, image1 in ref_images_dict.items():
    image1_preprocess = processor(images=image1, return_tensors="pt")['pixel_values'].to(device)
    ref_embeddings[key] = clip_model.get_image_features(image1_preprocess)

def predict_digit(image):
    # image2 = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    image2 = image
    image2 = cv2.resize(image2, (28, 28), interpolation=cv2.INTER_AREA)
  
    
    image2_preprocess = processor(images=image2, return_tensors="pt")['pixel_values'].to(device)
    image2_embedding = clip_model.get_image_features(image2_preprocess)

    # Calculate similarity scores
    scores_dict = {key: torch.nn.functional.cosine_similarity(ref_embedding, image2_embedding)
                   for key, ref_embedding in ref_embeddings.items()}

    # Find the label with the highest similarity score
    scores_dict_max_name = max(scores_dict, key=scores_dict.get)
    predicted_digit = string_to_digit[scores_dict_max_name]

    # scores_dict_pretty = {key: round(value.item(), 3) for key, value in scores_dict.items()}
    # print(scores_dict_pretty)
    plt.figure(figsize=(5, 5))
    plt.imshow(image2)
    plt.axis('off')
    plt.title(f'predicted_digit: {predicted_digit}')
    plt.show()
    
    return predicted_digit, image2

from skimage import io, filters
from skimage.color import rgb2gray

for page_num, row in rows:
    for cell in row.cells:
        x, y, w, h = cell.x, cell.y, cell.w, cell.h
        image = form_dict[page_num].image[y:y+h, x:x+w]
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        image_raw = image.copy()

        img = image
        img = rgb2gray(img)
        threshold = filters.threshold_otsu(img)  # Автоматический порог
        binary = img > threshold  # Делаем маску
        binary_cv2 = (binary * 255).astype(np.uint8)
        binary_cv2 = cv2.cvtColor(binary_cv2, cv2.COLOR_GRAY2BGR)
        predicted_digit, image2 = predict_digit(binary_cv2)
        # plt.figure(figsize=(1, 1))
        # plt.imshow(binary_cv2)
        # plt.axis('off')
        # plt.show()






In [11]:
n = 35
cur_samples = 1
num_gens = 3
sum = 0 
depth = 11

for depth in range(1, 20):
    cur_samples = 1
    for j in range(depth):
        for i in range(cur_samples):
            cur_samples += num_gens
    print(depth, cur_samples)

1 4
2 16
3 64
4 256
5 1024
6 4096
7 16384
8 65536
9 262144
10 1048576
11 4194304
12 16777216
13 67108864
14 268435456
15 1073741824


KeyboardInterrupt: 

In [12]:
35*1073/1000

37.555